In [319]:
import jpy_tools.parseSnake2 as jps
import pandas as pd

In [320]:
snakeFile = jps.SnakeFile()
snakeHeader = jps.SnakeHeader(snakeFile, "/datapool/data/Users/zhijian/projects/scumiatac/pipeline/config.yaml")

In [321]:
config = snakeHeader.getConfig()
snakeHeader

import pandas as pd
#configfile: "/datapool/data/Users/zhijian/projects/scumiatac/pipeline/config.yaml"
pipelineDir = config['pipelineDir']
resultDir = config["resultDir"].rstrip("/") + "/"
pipelineDir = config["pipelineDir"].rstrip("/") + "/"

In [322]:
pipelineDir = config['pipelineDir']
resultDir = config["resultDir"].rstrip("/") + "/"
pipelineDir = config["pipelineDir"].rstrip("/") + "/"

In [323]:
_ls = []
i = 1
for sample in config['sample'].keys():
    _ls_fq1 = []
    _ls_fq2 = []
    for fq1 in config['sample'][sample]['fq1']:
        fq2 = config['sample'][sample]['fq2'][config['sample'][sample]['fq1'].index(fq1)]
        _ls_fq1.append(fq1)
        _ls_fq2.append(fq2)
    _ls.append({
        'sample': sample,
        'fq1': _ls_fq1,
        'fq2': _ls_fq2,
    })
df_concat = pd.DataFrame(_ls)
df_concat = df_concat.assign(
    concatFq1=lambda _:  _['sample'] + ".R1.fastq.gz",
    concatFq2=lambda _:  _['sample'] + ".R2.fastq.gz",
)
df_concat = df_concat.set_index('sample')
df_concat

,fq1,fq2,concatFq1,concatFq2
sample,,,,
ATACP1000_1,[/datapool/data/Users/zhijian/projects/scumiat...,[/datapool/data/Users/zhijian/projects/scumiat...,ATACP1000_1.R1.fastq.gz,ATACP1000_1.R2.fastq.gz
ATACP1000_2,[/datapool/data/Users/zhijian/projects/scumiat...,[/datapool/data/Users/zhijian/projects/scumiat...,ATACP1000_2.R1.fastq.gz,ATACP1000_2.R2.fastq.gz
ATACP500_1,[/datapool/data/Users/zhijian/projects/scumiat...,[/datapool/data/Users/zhijian/projects/scumiat...,ATACP500_1.R1.fastq.gz,ATACP500_1.R2.fastq.gz
ATACP500_2,[/datapool/data/Users/zhijian/projects/scumiat...,[/datapool/data/Users/zhijian/projects/scumiat...,ATACP500_2.R1.fastq.gz,ATACP500_2.R2.fastq.gz
ATACSC,[/datapool/data/Users/zhijian/projects/scumiat...,[/datapool/data/Users/zhijian/projects/scumiat...,ATACSC.R1.fastq.gz,ATACSC.R2.fastq.gz


In [324]:
rule_concat = jps.SnakeRule(snakeFile, "concat_fastq", wildCard='sample', threads=1)
rule_concat.addCode("""
_ls = []
i = 1
for sample in config['sample'].keys():
    _ls_fq1 = []
    _ls_fq2 = []
    for fq1 in config['sample'][sample]['fq1']:
        fq2 = config['sample'][sample]['fq2'][config['sample'][sample]['fq1'].index(fq1)]
        _ls_fq1.append(fq1)
        _ls_fq2.append(fq2)
    _ls.append({
        'sample': sample,
        'fq1': _ls_fq1,
        'fq2': _ls_fq2,
    })
df_concat = pd.DataFrame(_ls)
df_concat = df_concat.assign(
    concatFq1=lambda _:  _['sample'] + ".R1.fastq.gz",
    concatFq2=lambda _:  _['sample'] + ".R2.fastq.gz",
)
df_concat = df_concat.set_index('sample')
df_concat
                    """)
rule_concat.addMetaDf('df_concat', ['concatFq1', 'concatFq2'], df_concat)
rule_concat.addMain('input', ['fq1', 'fq2'])
rule_concat.addMain('params', ['concatFq1', 'concatFq2'])
rule_concat.setShell(f"cat {{input.fq1}} > {{params.concatFq1}} && cat {{input.fq2}} > {{params.concatFq2}}")
rule_concat

2025-11-25 19:01:41.940 | INFO     | jpy_tools.parseSnake2:addRule:55 - concat_fastq step num: 1



## get parameter of rule `concat_fastq` ##
_ls = []
i = 1
for sample in config['sample'].keys():
    _ls_fq1 = []
    _ls_fq2 = []
    for fq1 in config['sample'][sample]['fq1']:
        fq2 = config['sample'][sample]['fq2'][config['sample'][sample]['fq1'].index(fq1)]
        _ls_fq1.append(fq1)
        _ls_fq2.append(fq2)
    _ls.append({
        'sample': sample,
        'fq1': _ls_fq1,
        'fq2': _ls_fq2,
    })
df_concat = pd.DataFrame(_ls)
df_concat = df_concat.assign(
    concatFq1=lambda _:  _['sample'] + ".R1.fastq.gz",
    concatFq2=lambda _:  _['sample'] + ".R2.fastq.gz",
)
df_concat = df_concat.set_index('sample')
df_concat
                    
for column in ['concatFq1', 'concatFq2']:
    df_concat[column] = resultDir + 'step1_concat_fastq/' + df_concat[column]
----------------
IN RULE
----------------
# parameter's dataframe of concat_fastq: 
# | sample      | fq1                                                                                                          

In [325]:
df_qc = df_concat[['concatFq1', 'concatFq2']].copy()
df_qc = df_qc.assign(
    qcr1=lambda _: _.index + ".R1_qc.fastq.gz",
    qcr2=lambda _: _.index + ".R2_qc.fastq.gz",
    html=lambda _: _.index + ".fastp.html",
    json=lambda _: _.index + ".fastp.json",
)
df_qc

,concatFq1,concatFq2,qcr1,qcr2,html,json
sample,,,,,,
ATACP1000_1,ATACP1000_1.R1.fastq.gz,ATACP1000_1.R2.fastq.gz,ATACP1000_1.R1_qc.fastq.gz,ATACP1000_1.R2_qc.fastq.gz,ATACP1000_1.fastp.html,ATACP1000_1.fastp.json
ATACP1000_2,ATACP1000_2.R1.fastq.gz,ATACP1000_2.R2.fastq.gz,ATACP1000_2.R1_qc.fastq.gz,ATACP1000_2.R2_qc.fastq.gz,ATACP1000_2.fastp.html,ATACP1000_2.fastp.json
ATACP500_1,ATACP500_1.R1.fastq.gz,ATACP500_1.R2.fastq.gz,ATACP500_1.R1_qc.fastq.gz,ATACP500_1.R2_qc.fastq.gz,ATACP500_1.fastp.html,ATACP500_1.fastp.json
ATACP500_2,ATACP500_2.R1.fastq.gz,ATACP500_2.R2.fastq.gz,ATACP500_2.R1_qc.fastq.gz,ATACP500_2.R2_qc.fastq.gz,ATACP500_2.fastp.html,ATACP500_2.fastp.json
ATACSC,ATACSC.R1.fastq.gz,ATACSC.R2.fastq.gz,ATACSC.R1_qc.fastq.gz,ATACSC.R2_qc.fastq.gz,ATACSC.fastp.html,ATACSC.fastp.json


In [326]:
rule_qc = jps.SnakeRule(snakeFile, "qc", 4, conda='scumiatac')
rule_qc.addCode(
    """
df_qc = df_concat[['concatFq1', 'concatFq2']].copy()
df_qc = df_qc.assign(
    qcr1=lambda _: _.index + ".R1_qc.fastq.gz",
    qcr2=lambda _: _.index + ".R2_qc.fastq.gz",
    html=lambda _: _.index + ".fastp.html",
    json=lambda _: _.index + ".fastp.json",
)
df_qc
"""
)
rule_qc.addMetaDf("df_qc", ["qcr1", "qcr2", "html", "json"], df_qc)
rule_qc.addMain("params", ["concatFq1", "concatFq2"], fromRule=rule_concat)
rule_qc.addMain("params", ["qcr1", "qcr2", "html", "json"])
rule_qc.setShell(
    """
fastp -i {params.concatFq1} -I {params.concatFq2} -o {params.qcr1} -O {params.qcr2} \
--html {params.html} --json {params.json} \
--thread {threads} -Q -L -A
"""
)
rule_qc

2025-11-25 19:01:49.312 | INFO     | jpy_tools.parseSnake2:addRule:55 - qc step num: 2



## get parameter of rule `qc` ##
df_qc = df_concat[['concatFq1', 'concatFq2']].copy()
df_qc = df_qc.assign(
    qcr1=lambda _: _.index + ".R1_qc.fastq.gz",
    qcr2=lambda _: _.index + ".R2_qc.fastq.gz",
    html=lambda _: _.index + ".fastp.html",
    json=lambda _: _.index + ".fastp.json",
)
df_qc
for column in ['qcr1', 'qcr2', 'html', 'json']:
    df_qc[column] = resultDir + 'step2_qc/' + df_qc[column]
----------------
IN RULE
----------------
# parameter's dataframe of qc: 
# | sample      | concatFq1               | concatFq2               | qcr1                       | qcr2                       | html                   | json                   |
# |:------------|:------------------------|:------------------------|:---------------------------|:---------------------------|:-----------------------|:-----------------------|
# | ATACP1000_1 | ATACP1000_1.R1.fastq.gz | ATACP1000_1.R2.fastq.gz | ATACP1000_1.R1_qc.fastq.gz | ATACP1000_1.R2_qc.fastq.gz | ATACP1000_1.fastp.html | ATACP100

In [327]:
df_umi = df_qc[[]]
df_umi = df_umi.assign(
    umifq1=lambda _: _.index + ".R1_umi.fastq.gz",
    umifq2=lambda _: _.index + ".R2_umi.fastq.gz",
    umilog=lambda _: _.index + ".umi.log",
    umierror=lambda _: _.index + ".umi.error",
    tempdir=lambda _: _.index + "_umi_tempdir/",
)
df_umi

,umifq1,umifq2,umilog,umierror,tempdir
sample,,,,,
ATACP1000_1,ATACP1000_1.R1_umi.fastq.gz,ATACP1000_1.R2_umi.fastq.gz,ATACP1000_1.umi.log,ATACP1000_1.umi.error,ATACP1000_1_umi_tempdir/
ATACP1000_2,ATACP1000_2.R1_umi.fastq.gz,ATACP1000_2.R2_umi.fastq.gz,ATACP1000_2.umi.log,ATACP1000_2.umi.error,ATACP1000_2_umi_tempdir/
ATACP500_1,ATACP500_1.R1_umi.fastq.gz,ATACP500_1.R2_umi.fastq.gz,ATACP500_1.umi.log,ATACP500_1.umi.error,ATACP500_1_umi_tempdir/
ATACP500_2,ATACP500_2.R1_umi.fastq.gz,ATACP500_2.R2_umi.fastq.gz,ATACP500_2.umi.log,ATACP500_2.umi.error,ATACP500_2_umi_tempdir/
ATACSC,ATACSC.R1_umi.fastq.gz,ATACSC.R2_umi.fastq.gz,ATACSC.umi.log,ATACSC.umi.error,ATACSC_umi_tempdir/


In [328]:
rule_umi = jps.SnakeRule(snakeFile, "umi", 1)
rule_umi.addCode(
    """
df_umi = df_qc[[]]
df_umi = df_umi.assign(
    umifq1=lambda _: _.index + ".R1_umi.fastq.gz",
    umifq2=lambda _: _.index + ".R2_umi.fastq.gz",
    umilog=lambda _: _.index + ".umi.log",
    umierror=lambda _: _.index + ".umi.error",
    tempdir=lambda _: _.index + "_umi_tempdir/",
)
df_umi
"""
)
rule_umi.addMetaDf("df_umi", ["umifq1", "umifq2" , "umilog", "umierror"])
rule_umi.addMain("params", ["qcr1", "qcr2"], rule_qc)
rule_umi.addMain("params", ["umifq1", "umifq2", "umilog", "umierror", "tempdir"])
rule_umi.setShell(
    """
umi_tools extract --extract-method=string --bc-pattern='CCCCCCCCNNNNNN' --bc-pattern2='CCCCCCCCNNNNNN' \
    -I {params.qcr1} -S {params.umifq1} --quality-filter-threshold=20 --quality-encoding='phred33' \
        --read2-in={params.qcr2}  --read2-out={params.umifq2}  \
            -L {params.umilog} -E {params.umierror} --temp-dir={params.tempdir} && rm {params.qcr1} {params.qcr2}
""")
rule_umi

2025-11-25 19:01:50.051 | INFO     | jpy_tools.parseSnake2:addRule:55 - umi step num: 3
2025-11-25 19:01:50.055 | WARNING  | jpy_tools.parseSnake2:addMetaDf:208 - please set `metaDf` if you want to record dataframe content in snakefile



## get parameter of rule `umi` ##
df_umi = df_qc[[]]
df_umi = df_umi.assign(
    umifq1=lambda _: _.index + ".R1_umi.fastq.gz",
    umifq2=lambda _: _.index + ".R2_umi.fastq.gz",
    umilog=lambda _: _.index + ".umi.log",
    umierror=lambda _: _.index + ".umi.error",
    tempdir=lambda _: _.index + "_umi_tempdir/",
)
df_umi
for column in ['umifq1', 'umifq2', 'umilog', 'umierror']:
    df_umi[column] = resultDir + 'step3_umi/' + df_umi[column]
----------------
IN RULE
----------------
rule umi:
    input:
        qcFinished = resultDir + 'step2_qc/' + '{sample}.finished',
    output:
        umiFinished = resultDir + 'step3_umi/' + '{sample}.finished',
    params:
        gpu = 0,
        qcr1 = lambda wildcard: df_qc.at[wildcard.sample, 'qcr1'],
        qcr2 = lambda wildcard: df_qc.at[wildcard.sample, 'qcr2'],
        umifq1 = lambda wildcard: df_umi.at[wildcard.sample, 'umifq1'],
        umifq2 = lambda wildcard: df_umi.at[wildcard.sample, 'umifq2'],
        umilog = lambda wildcar

In [329]:
df_trim = df_umi[['umifq1', 'umifq2']]
df_trim = df_trim.assign(
    base=lambda _: _.index + "_trimmed.fq.gz",
    fq1=lambda _: _.index + "_trimmed_1P.fq.gz",
    fq2=lambda _: _.index + "_trimmed_2P.fq.gz",
    finalFq1=lambda _: _.index + "_final_1P.fq.gz",
    finalFq2=lambda _: _.index + "_final_2P.fq.gz",
    finalBase=lambda _: _.index + "_final.fq.gz",
    adapterFa=config['adapterFa'],
    fpHtml=lambda _: _.index + "_fastp.html",
    fpJson=lambda _: _.index + "_fastp.json",
)
df_trim

,umifq1,umifq2,base,fq1,fq2,finalFq1,finalFq2,finalBase,adapterFa,fpHtml,fpJson
sample,,,,,,,,,,,
ATACP1000_1,ATACP1000_1.R1_umi.fastq.gz,ATACP1000_1.R2_umi.fastq.gz,ATACP1000_1_trimmed.fq.gz,ATACP1000_1_trimmed_1P.fq.gz,ATACP1000_1_trimmed_2P.fq.gz,ATACP1000_1_final_1P.fq.gz,ATACP1000_1_final_2P.fq.gz,ATACP1000_1_final.fq.gz,/datapool/data/Users/zhijian/projects/scumiata...,ATACP1000_1_fastp.html,ATACP1000_1_fastp.json
ATACP1000_2,ATACP1000_2.R1_umi.fastq.gz,ATACP1000_2.R2_umi.fastq.gz,ATACP1000_2_trimmed.fq.gz,ATACP1000_2_trimmed_1P.fq.gz,ATACP1000_2_trimmed_2P.fq.gz,ATACP1000_2_final_1P.fq.gz,ATACP1000_2_final_2P.fq.gz,ATACP1000_2_final.fq.gz,/datapool/data/Users/zhijian/projects/scumiata...,ATACP1000_2_fastp.html,ATACP1000_2_fastp.json
ATACP500_1,ATACP500_1.R1_umi.fastq.gz,ATACP500_1.R2_umi.fastq.gz,ATACP500_1_trimmed.fq.gz,ATACP500_1_trimmed_1P.fq.gz,ATACP500_1_trimmed_2P.fq.gz,ATACP500_1_final_1P.fq.gz,ATACP500_1_final_2P.fq.gz,ATACP500_1_final.fq.gz,/datapool/data/Users/zhijian/projects/scumiata...,ATACP500_1_fastp.html,ATACP500_1_fastp.json
ATACP500_2,ATACP500_2.R1_umi.fastq.gz,ATACP500_2.R2_umi.fastq.gz,ATACP500_2_trimmed.fq.gz,ATACP500_2_trimmed_1P.fq.gz,ATACP500_2_trimmed_2P.fq.gz,ATACP500_2_final_1P.fq.gz,ATACP500_2_final_2P.fq.gz,ATACP500_2_final.fq.gz,/datapool/data/Users/zhijian/projects/scumiata...,ATACP500_2_fastp.html,ATACP500_2_fastp.json
ATACSC,ATACSC.R1_umi.fastq.gz,ATACSC.R2_umi.fastq.gz,ATACSC_trimmed.fq.gz,ATACSC_trimmed_1P.fq.gz,ATACSC_trimmed_2P.fq.gz,ATACSC_final_1P.fq.gz,ATACSC_final_2P.fq.gz,ATACSC_final.fq.gz,/datapool/data/Users/zhijian/projects/scumiata...,ATACSC_fastp.html,ATACSC_fastp.json


In [330]:
rule_trim = jps.SnakeRule(snakeFile, "trim", 16, conda='scumiatac')
rule_trim.addCode(
    """
df_trim = df_umi[['umifq1', 'umifq2']]
df_trim = df_trim.assign(
    base=lambda _: _.index + "_trimmed.fq.gz",
    fq1=lambda _: _.index + "_trimmed_1P.fq.gz",
    fq2=lambda _: _.index + "_trimmed_2P.fq.gz",
    finalFq1=lambda _: _.index + "_final_1P.fq.gz",
    finalFq2=lambda _: _.index + "_final_2P.fq.gz",
    finalBase=lambda _: _.index + "_final.fq.gz",
    adapterFa=config['adapterFa'],
    fpHtml=lambda _: _.index + "_fastp.html",
    fpJson=lambda _: _.index + "_fastp.json",
)
df_trim
"""
)
rule_trim.addMetaDf('df_trim', ['base', 'fq1', 'fq2',  'finalFq1', 'finalFq2', 'finalBase', 'fpHtml', 'fpJson'], df_trim)
rule_trim.addMain('params', ['umifq1', 'umifq2'], rule_umi)
rule_trim.addMain('params', ['base', 'fq1', 'fq2', 'finalFq1', 'finalFq2', 'finalBase', 'adapterFa', 'fpHtml', 'fpJson'])
rule_trim.setShell(
    """
trimmomatic PE -Xmx16G -threads {threads} {params.umifq1} {params.umifq2} -baseout {params.base} HEADCROP:19 && rm {params.umifq1} {params.umifq2}
trimmomatic PE -Xmx16G -threads {threads} {params.fq1} {params.fq2} -baseout {params.finalBase} ILLUMINACLIP:{params.adapterFa}:2:30:11:8:true MINLEN:70 AVGQUAL:20
rm {params.fq1} {params.fq2}
fastp -i {params.finalFq1} -I {params.finalFq2} -o {params.fq1} -O {params.fq2} \
--html {params.fpHtml} --json {params.fpJson} \
--thread {threads} -Q -L -A
rm {params.fq1} {params.fq2}
""")
rule_trim

2025-11-25 19:01:50.512 | INFO     | jpy_tools.parseSnake2:addRule:55 - trim step num: 4



## get parameter of rule `trim` ##
df_trim = df_umi[['umifq1', 'umifq2']]
df_trim = df_trim.assign(
    base=lambda _: _.index + "_trimmed.fq.gz",
    fq1=lambda _: _.index + "_trimmed_1P.fq.gz",
    fq2=lambda _: _.index + "_trimmed_2P.fq.gz",
    finalFq1=lambda _: _.index + "_final_1P.fq.gz",
    finalFq2=lambda _: _.index + "_final_2P.fq.gz",
    finalBase=lambda _: _.index + "_final.fq.gz",
    adapterFa=config['adapterFa'],
    fpHtml=lambda _: _.index + "_fastp.html",
    fpJson=lambda _: _.index + "_fastp.json",
)
df_trim
for column in ['base', 'fq1', 'fq2', 'finalFq1', 'finalFq2', 'finalBase', 'fpHtml', 'fpJson']:
    df_trim[column] = resultDir + 'step4_trim/' + df_trim[column]
----------------
IN RULE
----------------
# parameter's dataframe of trim: 
# | sample      | umifq1                      | umifq2                      | base                      | fq1                          | fq2                          | finalFq1                   | finalFq2                   | 

In [331]:
df_bwa = df_trim[[]]
df_bwa = df_bwa.assign(
    bam = lambda _: _.index + ".bwa.bam",
    genome = config['genomeFa']
)
df_bwa

,bam,genome
sample,,
ATACP1000_1,ATACP1000_1.bwa.bam,/datapool/home/zhijian/data/wheat/genome/3.ref...
ATACP1000_2,ATACP1000_2.bwa.bam,/datapool/home/zhijian/data/wheat/genome/3.ref...
ATACP500_1,ATACP500_1.bwa.bam,/datapool/home/zhijian/data/wheat/genome/3.ref...
ATACP500_2,ATACP500_2.bwa.bam,/datapool/home/zhijian/data/wheat/genome/3.ref...
ATACSC,ATACSC.bwa.bam,/datapool/home/zhijian/data/wheat/genome/3.ref...


In [332]:
rule_bwa = jps.SnakeRule(snakeFile, "bwa", 32)
rule_bwa.addCode(
    """
df_bwa = df_trim[[]]
df_bwa = df_bwa.assign(
    bam = lambda _: _.index + ".bwa.bam",
    genome = config['genomeFa']
)
df_bwa
"""
)
rule_bwa.addMetaDf("df_bwa", ["bam"], df_bwa)
rule_bwa.addMain("params", ["finalFq1", "finalFq2"], rule_trim)
rule_bwa.addMain("params", ["genome", "bam"])
rule_bwa.setShell(
    """
bwa mem -t {threads} {params.genome} {params.finalFq1} {params.finalFq2} | \
samtools sort -@ {threads} -o {params.bam} -
samtools index -c {params.bam} -@ {threads}
""")
rule_bwa

2025-11-25 19:01:51.392 | INFO     | jpy_tools.parseSnake2:addRule:55 - bwa step num: 5



## get parameter of rule `bwa` ##
df_bwa = df_trim[[]]
df_bwa = df_bwa.assign(
    bam = lambda _: _.index + ".bwa.bam",
    genome = config['genomeFa']
)
df_bwa
for column in ['bam']:
    df_bwa[column] = resultDir + 'step5_bwa/' + df_bwa[column]
----------------
IN RULE
----------------
# parameter's dataframe of bwa: 
# | sample      | bam                 | genome                                                       |
# |:------------|:--------------------|:-------------------------------------------------------------|
# | ATACP1000_1 | ATACP1000_1.bwa.bam | /datapool/home/zhijian/data/wheat/genome/3.ref.mask.fasta.gz |
# | ATACP1000_2 | ATACP1000_2.bwa.bam | /datapool/home/zhijian/data/wheat/genome/3.ref.mask.fasta.gz |
# | ATACP500_1  | ATACP500_1.bwa.bam  | /datapool/home/zhijian/data/wheat/genome/3.ref.mask.fasta.gz |
# | ATACP500_2  | ATACP500_2.bwa.bam  | /datapool/home/zhijian/data/wheat/genome/3.ref.mask.fasta.gz |
# | ATACSC      | ATACSC.bwa.bam      | /datapool/home/zhi

In [333]:
df_bamQc = df_bwa[[]]
df_bamQc = df_bamQc.assign(
    bamqc=lambda _: _.index + ".qc.bam",
)
df_bamQc

,bamqc
sample,
ATACP1000_1,ATACP1000_1.qc.bam
ATACP1000_2,ATACP1000_2.qc.bam
ATACP500_1,ATACP500_1.qc.bam
ATACP500_2,ATACP500_2.qc.bam
ATACSC,ATACSC.qc.bam


In [388]:
rule_bamQc = jps.SnakeRule(snakeFile, "bam_qc", 16,)
rule_bamQc.addCode(
    """
df_bamQc = df_bwa[[]]
df_bamQc = df_bamQc.assign(
    bamqc=lambda _: _.index + ".qc.bam",
)
df_bamQc
"""
)
rule_bamQc.addMetaDf("df_bamQc", ["bamqc"], df_bamQc)
rule_bamQc.addMain("params", ["bam"], rule_bwa)
rule_bamQc.addMain("params", ["bamqc"])
rule_bamQc.setShell(
    """
samtools sort -n -@ {threads} {params.bam} -o {params.bamqc}.name_sorted.bam
python /datapool/data/Users/zhijian/projects/scumiatac/pipeline/scripts/bamFilterAndQc.py {params.bamqc}.name_sorted.bam {params.bamqc}.name_sorted.qc.bam
samtools sort -@ {threads} {params.bam}.name_sorted.qc.bam -o {params.bamqc}
samtool index -c {params.bamqc} -@ {threads}
rm {params.bamqc}.name_sorted.bam {params.bamqc}.name_sorted.qc.bam
""")
rule_bamQc

2025-11-26 13:30:47.957 | INFO     | jpy_tools.parseSnake2:addRule:55 - bam_qc step num: 8



## get parameter of rule `bam_qc` ##
df_bamQc = df_bwa[[]]
df_bamQc = df_bamQc.assign(
    bamqc=lambda _: _.index + ".qc.bam",
)
df_bamQc
for column in ['bamqc']:
    df_bamQc[column] = resultDir + 'step6_bam_qc/' + df_bamQc[column]
----------------
IN RULE
----------------
# parameter's dataframe of bam_qc: 
# | sample      | bamqc              |
# |:------------|:-------------------|
# | ATACP1000_1 | ATACP1000_1.qc.bam |
# | ATACP1000_2 | ATACP1000_2.qc.bam |
# | ATACP500_1  | ATACP500_1.qc.bam  |
# | ATACP500_2  | ATACP500_2.qc.bam  |
# | ATACSC      | ATACSC.qc.bam      |
rule bam_qc:
    input:
        bwaFinished = resultDir + 'step5_bwa/' + '{sample}.finished',
    output:
        bam_qcFinished = resultDir + 'step6_bam_qc/' + '{sample}.finished',
    params:
        gpu = 0,
        bam = lambda wildcard: df_bwa.at[wildcard.sample, 'bam'],
        bamqc = lambda wildcard: df_bamQc.at[wildcard.sample, 'bamqc'],
    threads:16
    priority:0
    shell:
        """
samtools sor

In [377]:
df_dedup = df_bamQc[[]]
df_dedup = df_dedup.assign(
    dedup_bam=lambda _: _.index + ".dedup.bam",
    family=lambda _: _.index + ".family.txt",
)
df_dedup

,dedup_bam,family
sample,,
ATACP1000_1,ATACP1000_1.dedup.bam,ATACP1000_1.family.txt
ATACP1000_2,ATACP1000_2.dedup.bam,ATACP1000_2.family.txt
ATACP500_1,ATACP500_1.dedup.bam,ATACP500_1.family.txt
ATACP500_2,ATACP500_2.dedup.bam,ATACP500_2.family.txt
ATACSC,ATACSC.dedup.bam,ATACSC.family.txt


In [378]:
rule_dedup = jps.SnakeRule(snakeFile, "dedup", 8, conda='scumiatac')
rule_dedup.addCode(
    """
df_dedup = df_bamQc[[]]
df_dedup = df_dedup.assign(
    dedup_bam=lambda _: _.index + ".dedup.bam",
    family=lambda _: _.index + ".family.txt",
)
df_dedup
"""
)
rule_dedup.addMetaDf("df_dedup", ["dedup_bam", "family"], df_dedup)
rule_dedup.addMain("params", ["bamqc"], rule_bamQc)
rule_dedup.addMain("params", ["dedup_bam", "family"])
rule_dedup.setShell("""       
_JAVA_OPTIONS="-Xmx16g" fgbio GroupReadsByUmi --input={params.bamqc} --output={params.bamqc}.gp.bam \
                     --strategy=adjacency --edits=1 --min-map-q=20 --threads {threads} --family-size-histogram={params.family}
__JAVA_OPTIONS="-Xmx16g" fgbio CallMolecularConsensusReads \
  --input={params.bamqc}.gp.bam \
  --output={params.dedup_bam} \
  --min-reads=5 \
  --min-input-base-quality=20 \
  --threads {threads}
rm {params.bamqc}.gp.bam           
""")
rule_dedup

2025-11-26 13:01:45.252 | INFO     | jpy_tools.parseSnake2:addRule:55 - dedup step num: 8



## get parameter of rule `dedup` ##
df_dedup = df_bamQc[[]]
df_dedup = df_dedup.assign(
    dedup_bam=lambda _: _.index + ".dedup.bam",
    family=lambda _: _.index + ".family.txt",
)
df_dedup
for column in ['dedup_bam', 'family']:
    df_dedup[column] = resultDir + 'step7_dedup/' + df_dedup[column]
----------------
IN RULE
----------------
# parameter's dataframe of dedup: 
# | sample      | dedup_bam             | family                 |
# |:------------|:----------------------|:-----------------------|
# | ATACP1000_1 | ATACP1000_1.dedup.bam | ATACP1000_1.family.txt |
# | ATACP1000_2 | ATACP1000_2.dedup.bam | ATACP1000_2.family.txt |
# | ATACP500_1  | ATACP500_1.dedup.bam  | ATACP500_1.family.txt  |
# | ATACP500_2  | ATACP500_2.dedup.bam  | ATACP500_2.family.txt  |
# | ATACSC      | ATACSC.dedup.bam      | ATACSC.family.txt      |
rule dedup:
    input:
        bam_qcFinished = resultDir + 'step6_bam_qc/' + '{sample}.finished',
    output:
        dedupFinished = resultDir + 'step

In [379]:
df_getFqAndMapping = df_dedup[[]]
df_getFqAndMapping = df_getFqAndMapping.assign(
    umiFq1=lambda _: _.index+'_1_fq.gz',
    umiFq2=lambda _: _.index+'_2_fq.gz',
    mapping=lambda _: _.index + '.mapping.bam',
    genomeFa=config['genomeFa']
)
df_getFqAndMapping

,umiFq1,umiFq2,mapping,genomeFa
sample,,,,
ATACP1000_1,ATACP1000_1_1_fq.gz,ATACP1000_1_2_fq.gz,ATACP1000_1.mapping.bam,/datapool/home/zhijian/data/wheat/genome/3.ref...
ATACP1000_2,ATACP1000_2_1_fq.gz,ATACP1000_2_2_fq.gz,ATACP1000_2.mapping.bam,/datapool/home/zhijian/data/wheat/genome/3.ref...
ATACP500_1,ATACP500_1_1_fq.gz,ATACP500_1_2_fq.gz,ATACP500_1.mapping.bam,/datapool/home/zhijian/data/wheat/genome/3.ref...
ATACP500_2,ATACP500_2_1_fq.gz,ATACP500_2_2_fq.gz,ATACP500_2.mapping.bam,/datapool/home/zhijian/data/wheat/genome/3.ref...
ATACSC,ATACSC_1_fq.gz,ATACSC_2_fq.gz,ATACSC.mapping.bam,/datapool/home/zhijian/data/wheat/genome/3.ref...


In [380]:
rule_umiMapping = jps.SnakeRule(snakeFile, "umiMapping", 16)
rule_umiMapping.addCode(
    """
df_getFqAndMapping = df_dedup[[]]
df_getFqAndMapping = df_getFqAndMapping.assign(
    umiFq1=lambda _: _.index+'_1_fq.gz',
    umiFq2=lambda _: _.index+'_2_fq.gz',
    mapping=lambda _: _.index + '.mapping.bam',
    genomeFa=config['genomeFa']
)
df_getFqAndMapping
"""
)
rule_umiMapping.addMetaDf("df_getFqAndMapping", ["umiFq1", "umiFq2", "mapping"], df_getFqAndMapping)
rule_umiMapping.addMain("params", ["dedup_bam"], rule_dedup)
rule_umiMapping.addMain('input', ['genomeFa'])
rule_umiMapping.addMain("params", ["umiFq1", "umiFq2", "mapping"])
rule_umiMapping.setShell(
    """
samtools fastq -@ {threads} -1 {params.umiFq1} -2 {params.umiFq2} -n {params.dedup_bam}
bwa mem -t {threads} {input.genomeFa} {params.umiFq1} {params.umiFq2} | samtools sort -@ {threads} -o {params.mapping} -
samtools index -c {params.mapping} -@ {threads}
""")
rule_umiMapping

2025-11-26 13:01:46.356 | INFO     | jpy_tools.parseSnake2:addRule:55 - umiMapping step num: 8



## get parameter of rule `umiMapping` ##
df_getFqAndMapping = df_dedup[[]]
df_getFqAndMapping = df_getFqAndMapping.assign(
    umiFq1=lambda _: _.index+'_1_fq.gz',
    umiFq2=lambda _: _.index+'_2_fq.gz',
    mapping=lambda _: _.index + '.mapping.bam',
    genomeFa=config['genomeFa']
)
df_getFqAndMapping
for column in ['umiFq1', 'umiFq2', 'mapping']:
    df_getFqAndMapping[column] = resultDir + 'step8_umiMapping/' + df_getFqAndMapping[column]
----------------
IN RULE
----------------
# parameter's dataframe of umiMapping: 
# | sample      | umiFq1              | umiFq2              | mapping                 | genomeFa                                                     |
# |:------------|:--------------------|:--------------------|:------------------------|:-------------------------------------------------------------|
# | ATACP1000_1 | ATACP1000_1_1_fq.gz | ATACP1000_1_2_fq.gz | ATACP1000_1.mapping.bam | /datapool/home/zhijian/data/wheat/genome/3.ref.mask.fasta.gz |
# | ATACP1000_2 |

In [381]:
snakeAll = jps.SnakeAll(snakeFile, rule_umiMapping)
snakeAll

rule all:
    input:
        umiMappingFinished = [resultDir + 'step8_umiMapping/' + "" + sample + ".finished" for sample in df_getFqAndMapping.index],

In [382]:
snakeFile.getMain('/datapool/data/Users/zhijian/projects/scumiatac/pipeline/snakefile')

import pandas as pd
#configfile: "/datapool/data/Users/zhijian/projects/scumiatac/pipeline/config.yaml"
pipelineDir = config['pipelineDir']
resultDir = config["resultDir"].rstrip("/") + "/"
pipelineDir = config["pipelineDir"].rstrip("/") + "/"


## get parameter of rule `concat_fastq` ##
_ls = []
i = 1
for sample in config['sample'].keys():
    _ls_fq1 = []
    _ls_fq2 = []
    for fq1 in config['sample'][sample]['fq1']:
        fq2 = config['sample'][sample]['fq2'][config['sample'][sample]['fq1'].index(fq1)]
        _ls_fq1.append(fq1)
        _ls_fq2.append(fq2)
    _ls.append({
        'sample': sample,
        'fq1': _ls_fq1,
        'fq2': _ls_fq2,
    })
df_concat = pd.DataFrame(_ls)
df_concat = df_concat.assign(
    concatFq1=lambda _:  _['sample'] + ".R1.fastq.gz",
    concatFq2=lambda _:  _['sample'] + ".R2.fastq.gz",
)
df_concat = df_concat.set_index('sample')
df_concat
                    
for column in ['concatFq1', 'concatFq2']:
    df_concat[column] = resultDir + 'step1_c